# Multi-Turn Chatbot Evaluations — Introduction & Setup

## Overview

This notebook introduces multi-turn evaluation concepts and builds the travel booking assistant that serves as the evaluation target throughout this module.

Evaluating single-turn agent interactions follows a pattern most teams understand well: provide an input, collect the output, judge the result. Frameworks like [Strands Evals SDK](https://github.com/strands-agents/evals) make this process systematic through evaluators that assess helpfulness, faithfulness, and tool usage. (See the [AWS blog post on Strands Evals](https://aws.amazon.com/blogs/machine-learning/evaluating-ai-agents-for-production-a-practical-guide-to-strands-evals/) for background.)

However, production conversations rarely stop at one turn. Real users engage in exchanges that unfold over multiple turns. They ask follow-up questions when answers are incomplete, change direction when new information surfaces, and express frustration when their needs go unmet. A travel assistant that handles "Book me a flight to Paris" well in isolation might struggle when the same user follows up with "Actually, can we look at trains instead?" or "What about hotels near the Eiffel Tower?"

The core difficulty is scale — you can't manually conduct hundreds of multi-turn conversations every time your agent changes, and writing scripted conversation flows locks you into predetermined paths that miss how real users behave.

## What You'll Learn

1. Why multi-turn evaluation is fundamentally harder than single-turn evaluation
2. Common failure modes unique to multi-turn systems (context forgetting, topic drift, goal abandonment, role breaking)
3. The three evaluation granularities: turn-level, session-level, output-level
4. The evaluation workflow: simulate → evaluate → analyze → iterate
5. How to set up the evaluation environment with Strands Agents, Strands Evals, and DeepEval
6. How to build a travel booking assistant with six tools as an evaluation target

## Prerequisites

- AWS account with Amazon Bedrock access (us-east-1)
- Access to Claude Sonnet 4 (agent model) and Amazon Nova Micro (evaluation judge) in Bedrock
- Python 3.10+
- AWS CLI configured with valid credentials

## 1. Why Multi-Turn Evaluation Is Fundamentally Harder

In a single-turn setting, every message is conditioned on a single request only. You can shuffle your test cases, evaluate them in any order, and each result stands on its own.

In a multi-turn conversation, **turn five is not just a response to message five — it is a response to the entire trajectory of messages one through four**. The assistant's answer in turn three shapes what the user asks in turn four, which determines what a good answer in turn five even looks like. Remove any single turn from that chain, and the rest may no longer make sense.

This interdependence has two practical consequences for evaluation:

1. **The search space is unpredictable.** You cannot predefine what the "correct" exchange should look like five turns deep. The "correct" next user utterance is a function of the agent's prior output. Static datasets of input-output pairs, regardless of scale, cannot model this property.
2. **Quality is cumulative, not per-turn.** A multi-turn system is judged as a session. One weak turn doesn't just lower an average — it can derail every turn that follows. Multi-turn metrics must capture trajectory-level consistency, not just turn-level correctness.

Manual testing can approximate real conversational dynamics, but it doesn't scale. Exercising every scenario, across every persona archetype, after every agent update demands effort that grows exponentially with the agent's expanding capabilities. In practice, coverage is sparse and unevenly distributed.

Some teams turn to prompt engineering as a shortcut — asking an LLM to "act like a user" during testing. Without structured persona definitions and explicit goal tracking, these approaches produce inconsistent results. The simulated user's behavior drifts between runs, making it difficult to compare evaluations over time or identify genuine regressions versus random variation.

A structured approach to user simulation bridges this gap by combining the realism of human conversation with the repeatability and scale of automated testing. That's what `ActorSimulator` (Strands Evals) and `ConversationSimulator` (DeepEval) provide.

### Common Failure Modes in Multi-Turn AI

Multi-turn conversations introduce failure modes that simply do not exist in single-turn systems. Recognizing them is a prerequisite for meaningful evaluation.

#### Context and Memory Failures

Most multi-turn breakdowns stem from poor context management:

| Failure | Description | Example |
|---------|-------------|--------|
| **Forgetting** | Agent loses information the user already provided | Asking for a name that was given four turns earlier |
| **Self-contradiction** | Agent contradicts its own prior statements | Recommending a product, then declaring it unavailable with no acknowledgment |
| **Thread confusion** | In multi-topic sessions, agent responds to the wrong thread | User asks about hotel after discussing flights; agent responds about flights |

#### Response Quality Failures

| Failure | Description | Example |
|---------|-------------|--------|
| **Ignoring latest message** | Response doesn't address what the user just said | Poor context window handling or retrieval |
| **Role violation** | Agent steps outside its assigned persona | A support agent offering medical guidance |
| **Partial resolution** | Only part of a compound request is addressed | User asks about price AND availability; agent only answers price |

#### Conversation Flow Failures

| Failure | Description | Example |
|---------|-------------|--------|
| **Infinite clarification loops** | Repeated questions without acting on answers | "What date?" → user answers → "And what date would you like?" |
| **Premature closure** | Conversation ends before the user's need is met | Agent says "Is there anything else?" after only partially helping |
| **Topic drift** | Gradually straying from the user's original intent | User wanted to book a flight; conversation drifts to travel tips |

### Evaluation Granularities

A robust multi-turn evaluation framework operates at three distinct levels. Each captures a different dimension of agent quality, and no single level is sufficient on its own.

| Level | Question It Answers | Strands Evals Evaluators | DeepEval Metrics |
|-------|-------------------|------------------------|------------------|
| **Turn-level** | "Given everything said so far, was this response appropriate?" | `CoherenceEvaluator`, `FaithfulnessEvaluator`, `HelpfulnessEvaluator`, `ResponseRelevanceEvaluator` | `TurnRelevancyMetric`, `KnowledgeRetentionMetric` |
| **Session-level** | "Did the conversation succeed as a whole?" | `GoalSuccessRateEvaluator`, `InteractionsEvaluator` | `ConversationCompletenessMetric`, `GoalAccuracyMetric` |
| **Output-level** | "Is this response well-formed?" | `OutputEvaluator` | `RoleAdherenceMetric`, `TopicAdherenceMetric` |

A comprehensive evaluation pipeline layers all three: output-level checks as a baseline filter, turn-level scoring for diagnostic precision, and session-level assessment as the ultimate measure of agent effectiveness.

### The Evaluation Workflow

The core loop for multi-turn evaluation:

```
1. Define scenarios  →  What should the conversation accomplish? What user persona?
2. Simulate conversations  →  ActorSimulator (Strands) or ConversationSimulator (DeepEval)
3. Evaluate with metrics  →  Trace-level + session-level + output-level evaluators
4. Analyze results  →  Identify failure patterns, compare across scenarios
5. Iterate  →  Improve prompt/tools/logic, re-evaluate
```

This module walks through each step with hands-on code.

## 2. Setting Up the Environment

We need three packages:

| Package | Purpose | Used For |
|---------|---------|----------|
| `strands-agents` | Agent framework | Building the travel booking assistant |
| `strands-agents-evals` | Evaluation SDK | `ActorSimulator`, `ToolSimulator`, `ExperimentGenerator`, all evaluators |
| `deepeval` | Complementary eval framework | `ConversationSimulator`, multi-turn specific metrics |

Both Strands Evals and DeepEval use Amazon Bedrock models as LLM judges. The agent under test uses Claude Sonnet 4, while evaluation judges use Amazon Nova Micro for cost efficiency.

In [ ]:
%pip install -q -r requirements.txt

### Verify AWS Credentials

All models run on Amazon Bedrock. Verify your credentials are configured correctly.

In [ ]:
import boto3

sts = boto3.client('sts')
identity = sts.get_caller_identity()
print(f'Account: {identity["Account"]}')
print(f'ARN:     {identity["Arn"]}')

### Verify Model Access

We use two models throughout this module:

| Model | ID | Role |
|-------|-----|------|
| Claude Sonnet 4 | `us.anthropic.claude-sonnet-4-20250514-v1:0` | Powers the travel booking agent |
| Amazon Nova Micro | `amazon.nova-micro-v1:0` | Evaluation judge (cost-efficient for scoring) |

In [ ]:
bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

for model_id in ['us.anthropic.claude-sonnet-4-20250514-v1:0', 'amazon.nova-micro-v1:0']:
    try:
        resp = bedrock.converse(
            modelId=model_id,
            messages=[{'role': 'user', 'content': [{'text': 'Say hello in one word.'}]}],
            inferenceConfig={'maxTokens': 10}
        )
        text = resp['output']['message']['content'][0]['text']
        print(f'  OK {model_id}: {text}')
    except Exception as e:
        print(f'  ERROR {model_id}: {e}')

### Import Core Modules

Import the key classes we'll use throughout the module. These are organized into four categories:

- **Agent framework**: `Agent`, `tool` — for building the chatbot
- **Evaluation core**: `Case`, `Experiment` — for defining test cases and running evaluations
- **Evaluators**: `HelpfulnessEvaluator`, `GoalSuccessRateEvaluator`, etc. — for scoring conversations
- **Simulation & telemetry**: `ActorSimulator`, `StrandsEvalsTelemetry` — for generating and capturing conversations

In [ ]:
# Agent framework
from strands import Agent, tool
from datetime import date

# Evaluation core
from strands_evals import Case, Experiment

# Evaluators — trace-level (per-turn) and session-level (full conversation)
from strands_evals.evaluators import (
    HelpfulnessEvaluator,      # 7-level helpfulness scoring
    CoherenceEvaluator,        # 5-level logical cohesion scoring
    FaithfulnessEvaluator,     # 5-level groundedness scoring
    ResponseRelevanceEvaluator,# 5-level relevance scoring
    GoalSuccessRateEvaluator,  # Binary goal success (basic + assertion modes)
    OutputEvaluator,           # Custom rubric-based evaluation
    TrajectoryEvaluator,       # Tool usage sequence evaluation
)

# Simulation
from strands_evals import ActorSimulator
from strands_evals.simulation.tool_simulator import ToolSimulator

# Telemetry for trace capture
from strands_evals.telemetry import StrandsEvalsTelemetry
from strands_evals.mappers import StrandsInMemorySessionMapper

# Extractors for trajectory analysis
from strands_evals.extractors import tools_use_extractor

print('All imports successful.')

## 3. Building the Travel Booking Assistant

We build a travel booking assistant with [Strands Agents](https://strandsagents.com/) that handles the end-to-end workflow of planning and managing travel: searching for flights and hotels, placing bookings, retrieving existing reservations, and processing cancellations.

This application makes a strong evaluation target because it naturally demands multi-turn interaction. A user rarely searches, selects, and books in a single message. Instead, a typical session might span half a dozen turns: asking for flights, narrowing options by price, switching to hotel search, booking both, and then circling back to modify a reservation. That conversational depth is exactly what makes multi-turn evaluation essential.

### Design Choices

| Choice | Rationale |
|--------|----------|
| **Hardcoded mock data** | Search results come from static data so variability in evaluation results stems from the LLM's conversational behavior, not external dependencies |
| **In-memory booking state** | Bookings persist across turns within a session via a simple dictionary |
| **Six tools** | Covers the full CRUD lifecycle: search, create, read, delete |
| **`callback_handler=None`** | Suppresses streaming output to avoid Rich/Jupyter rendering conflicts |

### Tool Definitions

Each tool uses the `@tool` decorator from Strands Agents, which automatically extracts the function signature, type hints, and docstring to create the tool specification the LLM uses for tool selection.

In [ ]:
from strands import Agent, tool
from datetime import date

bookings: dict = {}
booking_counter = 0

@tool
def search_flights(origin: str, destination: str, departure_date: str, return_date: str = None) -> str:
    """Search for available flights between two cities.

    Args:
        origin: Departure city or airport code (e.g. 'New York' or 'JFK')
        destination: Arrival city or airport code (e.g. 'London' or 'LHR')
        departure_date: Departure date in YYYY-MM-DD format
        return_date: Optional return date in YYYY-MM-DD format for round trips
    """
    flights = [
        {"flight": "AA101", "airline": "American Airlines", "departs": "08:00", "arrives": "20:00", "price": 450, "class": "Economy"},
        {"flight": "BA202", "airline": "British Airways",   "departs": "11:30", "arrives": "23:45", "price": 620, "class": "Economy"},
        {"flight": "UA303", "airline": "United Airlines",   "departs": "14:00", "arrives": "02:15", "price": 890, "class": "Business"},
    ]
    trip = f"round-trip (return: {return_date})" if return_date else "one-way"
    lines = [f"Flights from {origin} to {destination} on {departure_date} ({trip}):"]
    for f in flights:
        lines.append(f"  {f['flight']} | {f['airline']} | {f['departs']}-{f['arrives']} | ${f['price']} | {f['class']}")
    return "\n".join(lines)

@tool
def search_hotels(city: str, check_in: str, check_out: str, guests: int = 1) -> str:
    """Search for available hotels in a city.

    Args:
        city: Destination city
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
        guests: Number of guests (default 1)
    """
    hotels = [
        {"name": "Grand Plaza Hotel",  "stars": 5, "price_per_night": 320, "amenities": "Pool, Spa, Restaurant"},
        {"name": "City Center Inn",     "stars": 3, "price_per_night": 95,  "amenities": "Free WiFi, Breakfast"},
        {"name": "Boutique Stay",       "stars": 4, "price_per_night": 180, "amenities": "Gym, Bar, Concierge"},
    ]
    nights = (date.fromisoformat(check_out) - date.fromisoformat(check_in)).days
    lines = [f"Hotels in {city} | {check_in} to {check_out} ({nights} nights, {guests} guest(s)):"]
    for h in hotels:
        total = h["price_per_night"] * nights
        lines.append(f"  {h['name']} ({h['stars']}*) | ${h['price_per_night']}/night (${total} total) | {h['amenities']}")
    return "\n".join(lines)

@tool
def book_flight(flight_number: str, passenger_name: str, origin: str, destination: str, travel_date: str) -> str:
    """Book a flight for a passenger.

    Args:
        flight_number: Flight number to book (e.g. 'AA101')
        passenger_name: Full name of the passenger
        origin: Departure city or airport
        destination: Arrival city or airport
        travel_date: Travel date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"FLT-{booking_counter:04d}"
    bookings[ref] = {"type": "flight", "flight": flight_number, "passenger": passenger_name,
                     "route": f"{origin} -> {destination}", "date": travel_date, "status": "Confirmed"}
    return f"Flight booked! Ref: {ref} | {flight_number} | {origin} -> {destination} on {travel_date} | Passenger: {passenger_name}"

@tool
def book_hotel(hotel_name: str, guest_name: str, city: str, check_in: str, check_out: str) -> str:
    """Book a hotel room for a guest.

    Args:
        hotel_name: Name of the hotel to book
        guest_name: Full name of the guest
        city: City where the hotel is located
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"HTL-{booking_counter:04d}"
    bookings[ref] = {"type": "hotel", "hotel": hotel_name, "guest": guest_name,
                     "city": city, "check_in": check_in, "check_out": check_out, "status": "Confirmed"}
    return f"Hotel booked! Ref: {ref} | {hotel_name} in {city} | {check_in} to {check_out} | Guest: {guest_name}"

@tool
def get_bookings() -> str:
    """Retrieve all current bookings."""
    if not bookings:
        return "No bookings found."
    lines = ["Current bookings:"]
    for ref, b in bookings.items():
        if b["type"] == "flight":
            lines.append(f"  [{ref}] Flight {b['flight']} | {b['route']} on {b['date']} | {b['passenger']} | {b['status']}")
        else:
            lines.append(f"  [{ref}] Hotel {b['hotel']} in {b['city']} | {b['check_in']} to {b['check_out']} | {b['guest']} | {b['status']}")
    return "\n".join(lines)

@tool
def cancel_booking(booking_ref: str) -> str:
    """Cancel an existing booking.

    Args:
        booking_ref: Booking reference number (e.g. 'FLT-0001' or 'HTL-0002')
    """
    if booking_ref not in bookings:
        return f"Booking {booking_ref} not found."
    bookings[booking_ref]["status"] = "Cancelled"
    return f"Booking {booking_ref} has been cancelled."

ALL_TOOLS = [search_flights, search_hotels, book_flight, book_hotel, get_bookings, cancel_booking]

SYSTEM_PROMPT = (
    "You are a helpful travel booking assistant. You help users search for flights and hotels, "
    "make bookings, view existing reservations, and cancel bookings. "
    "Always confirm details with the user before completing a booking. "
    "Use today's date as context when dates are not fully specified."
)

print(f"Defined {len(ALL_TOOLS)} tools: {[t.__name__ for t in ALL_TOOLS]}")


### Creating the Agent

The agent uses Claude Sonnet 4 as its underlying model. The system prompt defines its role, behavioral boundaries, and the instruction to confirm details before booking — a behavior we can later evaluate with metrics.

In [ ]:
agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    tools=ALL_TOOLS,
    callback_handler=None,
)

print('Travel booking agent created.')

### Baseline Multi-Turn Conversation

Before we automate evaluation, let's run a realistic multi-turn conversation manually. This establishes the baseline behavior we expect and demonstrates the kind of context-dependent interactions that multi-turn evaluation needs to verify systematically.

This scenario involves a user who:
1. Searches for flights (requires `search_flights`)
2. Books the cheapest option (requires `book_flight` — agent must remember the search results)
3. Switches to hotel search (requires `search_hotels` — agent must remember the destination)
4. Books a hotel (requires `book_hotel` — agent must carry forward the guest name)
5. Reviews all bookings (requires `get_bookings` — agent must retrieve the session state)
6. Cancels the hotel (requires `cancel_booking` — agent must use the correct reference)

In [ ]:
# Reset state for a clean demo
bookings.clear()
booking_counter = 0

demo_agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    tools=ALL_TOOLS,
    callback_handler=None,
)

# A realistic 6-turn conversation that exercises context retention,
# tool selection, and state management across turns
turns = [
    "I'm looking for flights from New York to London on September 1st, 2025.",
    "Book the cheapest one for Jane Doe.",
    "Now find me a hotel in London for September 1-5, 2025.",
    "Book the City Center Inn for Jane Doe.",
    "Show me all my bookings.",
    "Actually, cancel the hotel booking.",
]

for i, user_msg in enumerate(turns, 1):
    print(f'\n{"="*70}')
    print(f'Turn {i} [User]: {user_msg}')
    print(f'{"="*70}')
    response = demo_agent(user_msg)
    print(f'[Assistant]: {response}')

### What to Observe

In the conversation above, pay attention to how the agent:

- **Maintains context across turns**: Remembers "London" and "Jane Doe" without being reminded
- **Selects tools appropriately**: Uses `search_flights` → `book_flight` → `search_hotels` → `book_hotel` → `get_bookings` → `cancel_booking` in the correct sequence
- **Manages state**: The booking references from earlier turns are available in later turns
- **Confirms before booking**: Follows the system prompt instruction (or doesn't — which is something we'd catch in evaluation)

These are the behaviors we want to evaluate systematically across hundreds of scenarios. In the next notebooks, we'll:

| Notebook | What It Does |
|----------|-------------|
| **02 — Strands Simulation** | Simulate conversations automatically with `ActorSimulator` |
| **03 — DeepEval Simulation** | Simulate with `ConversationSimulator` and diverse personas *(Ester)* |
| **04 — DeepEval Metrics** | Evaluate with DeepEval's multi-turn metrics *(Ester)* |
| **05 — Strands Evaluators** | Evaluate with Strands trace-level and session-level evaluators |
| **06 — Synthetic Data** | Generate diverse test scenarios with `ExperimentGenerator` |
| **07 — Tool Simulation** | Use `ToolSimulator` for LLM-powered tool responses |
| **08 — E2E Pipeline** | Combine everything into a production evaluation pipeline |

## Next Steps

Continue to **`04-11-02-strands-simulation.ipynb`** to learn how to simulate multi-turn conversations automatically with Strands `ActorSimulator`.